# Option data pre-processing with MARKETDATA API

MarketData API를 이용해서, SPX 옵션체인 데이터를 불러온다.

데이터를 불러온 이후에는, 현재 프로젝트의 용도에 맞게 데이터를 전처리 한다.

**Credit을 아끼는 핵심 원칙**


MarketData의 현재 pricing 구조상 option chain은 데이터 종류에 따라 비용이 크게 다릅니다. Historical query에서 date를 지정하면 1,000 option symbols당 1 credit이지만, current real-time/15-minute delayed chain은 symbol당 1 credit입니다.

따라서 우리 프로젝트에서는 다음 원칙을 지키면 됩니다.

- 항상 date=를 넣어서 historical chain을 요청한다.
- expiration=all을 함부로 사용하지 않는다.
- dte로 한 expiry만 받는다.
- strikeLimit으로 ATM 주변만 받는다.
- Call/Put은 한 요청에서 같이 받는다.
- API 호출 전 CSV 존재 여부를 확인한다.
- 처음에는 strikeLimit=10, 구조 확인 후 40~60으로 늘린다

In [5]:
import os

TOKEN = os.environ.get("MARKETDATA_TOKEN", "No token found")

print("Token loaded:", bool(TOKEN) and TOKEN != "No token found")

Token loaded: True


In [3]:
import requests
import pandas as pd
import os
from pathlib import Path

먼저 테스트용으로, 최초 API요청을 한번 한다.  
- 사용날짜 : 2026-07-15
- Target maturity : 30DTE
- Strike : ATM 주변 10개 데이터

dte=30 는 historical date 기준 30일에 가장 가까운 단일 만기를 사용한다. 


In [6]:
import requests

url = "https://api.marketdata.app/v1/options/chain/SPX/"

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/json",
}

params = {
    "date": "2026-07-15",
    "dte": 30,
    "am": "false",
    "pm": "true",
    "strikeLimit": 10,
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=60,
)

print("HTTP status:", response.status_code)

HTTP status: 203


요청한 API가 적절하게 들어왔는지 key를 확인해본다.그리고 옵션의 개수도 확인한다.

In [9]:
data = response.json()

print(data.keys())
print("API status:", data.get("s"))
print("Number of contracts:", len(data["optionSymbol"]))

dict_keys(['s', 'optionSymbol', 'underlying', 'expiration', 'side', 'strike', 'firstTraded', 'dte', 'updated', 'bid', 'bidSize', 'mid', 'ask', 'askSize', 'last', 'openInterest', 'volume', 'inTheMoney', 'intrinsicValue', 'extrinsicValue', 'underlyingPrice', 'iv', 'delta', 'gamma', 'theta', 'vega'])
API status: ok
Number of contracts: 20


이제 pandas를 이용하여 데이터 프레임을 만든다.

In [15]:
columns = {
    key: value
    for key, value in data.items()
    if isinstance(value, list)
}

df = pd.DataFrame(columns)


print(df.shape)
print(df.columns.tolist())
check_cols = [
    "optionSymbol",
    "expiration",
    "dte",
    "side",
    "strike",
    "bid",
    "ask",
    "mid",
    "volume",
    "openInterest",
    "underlyingPrice",
    "iv"
]

df[check_cols].head(20)

df.insert(0, "quoteDate", "2026-07-15")

df.head()

(20, 25)
['optionSymbol', 'underlying', 'expiration', 'side', 'strike', 'firstTraded', 'dte', 'updated', 'bid', 'bidSize', 'mid', 'ask', 'askSize', 'last', 'openInterest', 'volume', 'inTheMoney', 'intrinsicValue', 'extrinsicValue', 'underlyingPrice', 'iv', 'delta', 'gamma', 'theta', 'vega']


,quoteDate,optionSymbol,underlying,expiration,side,strike,firstTraded,dte,updated,bid,...,volume,inTheMoney,intrinsicValue,extrinsicValue,underlyingPrice,iv,delta,gamma,theta,vega
0,2026-07-15,SPXW260814C07550000,SPX,1786737600,call,7550,1781184600,30,1784145600,137.2,...,52,True,22.4102,115.0898,7572.4102,None,None,None,None,None
1,2026-07-15,SPXW260814C07555000,SPX,1786737600,call,7555,1783949400,30,1784145600,134.0,...,7,True,17.4102,116.8398,7572.4102,None,None,None,None,None
2,2026-07-15,SPXW260814C07560000,SPX,1786737600,call,7560,1782135000,30,1784145600,130.7,...,93,True,12.4102,118.6398,7572.4102,None,None,None,None,None
3,2026-07-15,SPXW260814C07565000,SPX,1786737600,call,7565,1783949400,30,1784145600,127.4,...,55,True,7.4102,120.2898,7572.4102,None,None,None,None,None
4,2026-07-15,SPXW260814C07570000,SPX,1786737600,call,7570,1782135000,30,1784145600,124.2,...,120,True,2.4102,122.0898,7572.4102,None,None,None,None,None


In [18]:
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"],
        text=True
    ).strip()
)

print(PROJECT_ROOT)

/home/minseok/finance_projects/option-pricing-volatility


이제, API를 통해서 내려받은 데이터를 raw csv로 다운로드한다.

In [22]:
RAW_DIR = PROJECT_ROOT / "data/raw/marketdata_spx"

RAW_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


In [23]:
file_path = (
    RAW_DIR
    / "SPX_2026-07-15_dte030_pm_sl010.csv"
)
# dte = 30, pm, strikeLimit = 10

df.to_csv(
    file_path,
    index=False,
)

print("Saved:", file_path)

test_df = pd.read_csv(file_path)

print(test_df.shape)
test_df.head()

Saved: /home/minseok/finance_projects/option-pricing-volatility/data/raw/marketdata_spx/SPX_2026-07-15_dte030_pm_sl010.csv
(20, 26)


,quoteDate,optionSymbol,underlying,expiration,side,strike,firstTraded,dte,updated,bid,...,volume,inTheMoney,intrinsicValue,extrinsicValue,underlyingPrice,iv,delta,gamma,theta,vega
0,2026-07-15,SPXW260814C07550000,SPX,1786737600,call,7550,1781184600,30,1784145600,137.2,...,52,True,22.4102,115.0898,7572.4102,NaN,NaN,NaN,NaN,NaN
1,2026-07-15,SPXW260814C07555000,SPX,1786737600,call,7555,1783949400,30,1784145600,134.0,...,7,True,17.4102,116.8398,7572.4102,NaN,NaN,NaN,NaN,NaN
2,2026-07-15,SPXW260814C07560000,SPX,1786737600,call,7560,1782135000,30,1784145600,130.7,...,93,True,12.4102,118.6398,7572.4102,NaN,NaN,NaN,NaN,NaN
3,2026-07-15,SPXW260814C07565000,SPX,1786737600,call,7565,1783949400,30,1784145600,127.4,...,55,True,7.4102,120.2898,7572.4102,NaN,NaN,NaN,NaN,NaN
4,2026-07-15,SPXW260814C07570000,SPX,1786737600,call,7570,1782135000,30,1784145600,124.2,...,120,True,2.4102,122.0898,7572.4102,NaN,NaN,NaN,NaN,NaN


## raw csv 존재시 추가 API 호출 방지 코드

재사용 가능한 `download_spx_chain`은 `src/option_pricing_volatility/market_data/`로 승격했다. 아래에서는 함수 구현을 복사하지 않고 패키지에서 import한다. 함수는 nonempty raw CSV가 있으면 인증정보를 확인하거나 API를 호출하지 않는다.

In [ ]:
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"],
        text=True,
    ).strip()
)
SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from option_pricing_volatility.market_data import download_spx_chain

RAW_DIR = PROJECT_ROOT / "data/raw/marketdata_spx"

In [ ]:
# 추가 API 호출 방지 예시

df = download_spx_chain(
    quote_date="2026-07-15",
    target_dte=30,
    strike_limit=10,
    raw_dir=RAW_DIR,
)